# Build DL-based pose ranker from MD trajectories.

This notebook shows the training of a DL-based model to predict pose quality, trained on MD snapshots.

In [19]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINConv, global_mean_pool
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader

import MDAnalysis as mda
from MDAnalysis.topology.guessers import guess_bonds
from MDAnalysis.analysis import rms, distances

from rdkit import Chem
from rdkit.Chem import Descriptors, rdPartialCharges

import itertools
import warnings
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

import pathlib
from pathlib import Path
from rdkit import RDLogger
from tqdm.notebook import tqdm

# Get the main logger
logger = RDLogger.logger()

# Set its level to ERROR so only serious messages show up
logger.setLevel(RDLogger.ERROR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#verify Cuda is actually working

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (runtime):", torch.version.cuda)

print("Torch CUDA libraries loaded:")
for lib in torch.cuda._CudaBase.__module__.split(":"):
    print(lib)

Torch version: 2.8.0+cu128
CUDA available: True
CUDA version (runtime): 12.8
Torch CUDA libraries loaded:
torch.cuda


## Setup.

In [20]:
readout = 'trajs'

In [21]:
# define paths
HERE = Path(pathlib.Path.cwd())
DATA = HERE / f"data_{readout}"
DATA.mkdir(parents=True, exist_ok=True)


In [23]:
def atomic_number(atom):
    try:
        return Chem.GetPeriodicTable().GetAtomicNumber(atom.element)
    except Exception:
        return 0

def load_trajectory_as_graphs(pdb_path, traj_path, ligand_resname="UNK"):
    try:
        u = mda.Universe(pdb_path, traj_path)
    except Exception as e:
        warnings.warn(f"Failed to load Universe from {pdb_path} + {traj_path}: {e}")
        return []

    # --- Select ligand atoms ---
    ligand = u.select_atoms(f"resname {ligand_resname}")
    if len(ligand) == 0:
        warnings.warn(f"No ligand atoms found in {pdb_path} (resname {ligand_resname}); skipping.")
        return []

    # --- Ensure element info ---
    for atom in ligand:
        if atom.element is None and atom.name:
            atom.element = atom.name[0].upper()

    # --- Guess bonds ---
    try:
        ligand.guess_bonds()
    except Exception as e:
        warnings.warn(f"Failed to guess bonds for {pdb_path}: {e}")
        return []

    # --- Build graphs for each frame ---
    graphs = []

    # Build a mapping from global atom index → local ligand index
    atom_index_map = {atom.index: i for i, atom in enumerate(ligand.atoms)}

    for ts in u.trajectory:
        coords = ligand.positions  # shape (N_atoms, 3)
        z = torch.tensor([atomic_number(atom) for atom in ligand], dtype=torch.long)

        edge_index = []
        for bond in ligand.bonds:
            try:
                i, j = [atom_index_map[a.index] for a in bond.atoms]
                edge_index.append([i, j])
                edge_index.append([j, i])
            except KeyError:
                # Skip bonds referencing atoms outside ligand (shouldn’t happen but safe)
                continue

        # Convert to PyTorch tensor
        if len(edge_index) == 0:
            warnings.warn(f"No bonds found in {pdb_path}, skipping frame.")
            continue

        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        pos = torch.tensor(coords, dtype=torch.float)

        graphs.append(Data(z=z, edge_index=edge_index, pos=pos))

    return graphs

def compute_pose_metrics(u, ligand_sel="resname UNK", ref_frame=0, cutoff=4.5):
    """
    Compute ΔRMSD, ΔE proxy, and f_native for each trajectory frame.
    Returns a list of [ΔRMSD, ΔE, f_native].
    """

    # --- selections ---
    ligand = u.select_atoms(ligand_sel)
    protein = u.select_atoms("protein")
    if len(ligand) == 0 or len(protein) == 0:
        raise ValueError("Could not find ligand or protein atoms")

    # --- reference coordinates (bound pose) ---
    ref_coords = ligand.positions.copy()

    # --- native contacts in reference ---
    ref_dists = distances.distance_array(ref_coords, protein.positions)
    ref_contacts = ref_dists < cutoff

    metrics = []

    for ts in u.trajectory:
        # ΔRMSD (Å)
        drmsd = rms.rmsd(ligand.positions, ref_coords, superposition=True)

        # --- approximate interaction energy proxy ---
        # Coulomb + 6-12 Lennard-Jones like term (very rough)
        dist = distances.distance_array(ligand.positions, protein.positions)
        inv_dist = np.clip(1.0 / dist, 0, 1e2)  # avoid div/0
        # simple proxy: attractive - repulsive
        energy_proxy = np.sum(-inv_dist**6 + inv_dist**12)

        # --- fraction of native contacts ---
        contacts = dist < cutoff
        fnat = (contacts & ref_contacts).sum() / (ref_contacts.sum() + 1e-8)

        metrics.append([drmsd, energy_proxy, fnat])

    return np.array(metrics)

def attach_metrics_to_graphs(pdb_path, traj_path, graphs, ligand_resname="UNK"):
    u = mda.Universe(pdb_path, traj_path)
    metrics = compute_pose_metrics(u, ligand_sel=f"resname {ligand_resname}")

    n = min(len(metrics), len(graphs))
    for i in range(n):
        graphs[i].y = torch.tensor(metrics[i], dtype=torch.float32).unsqueeze(0)

    return graphs


def load_all_targets(base_dir, ligand_resname="UNK"):
    all_graphs = []
    for target in os.listdir(base_dir):
        target_path = os.path.join(base_dir, target)
        if not os.path.isdir(target_path):
            continue

        for pdb_struct in os.listdir(target_path):
            output_dir = os.path.join(target_path, pdb_struct, "output")
            pdb_path = os.path.join(output_dir, "topology.pdb")
            traj_path = os.path.join(output_dir, "trajectory.dcd")

            # Skip missing or empty files
            if not (os.path.exists(pdb_path) and os.path.exists(traj_path)):
                warnings.warn(f"Missing topology or trajectory for {pdb_struct}; skipping.")
                continue
            if os.path.getsize(traj_path) == 0:
                warnings.warn(f"Empty trajectory file for {pdb_struct}; skipping.")
                continue

            try:
                graphs = load_trajectory_as_graphs(pdb_path, traj_path, ligand_resname)
                if graphs:
                    graphs = attach_metrics_to_graphs(pdb_path, traj_path, graphs, ligand_resname)
                    all_graphs.extend(graphs)
            except Exception as e:
                warnings.warn(f"Failed to process {pdb_struct}: {e}")
                continue

    print(f"Loaded {len(all_graphs)} graphs from trajectories.")
    return all_graphs


In [24]:
# Example usage:
base_dir = DATA
graphs = load_all_targets(base_dir)

print(f"Loaded {len(graphs)} graphs from trajectories")

/tmp/ipykernel_10279/3250331581.py:129: UserWarning: Missing topology or trajectory for 4WMR; skipping.
  warnings.warn(f"Missing topology or trajectory for {pdb_struct}; skipping.")
/home/corey/miniconda3/envs/docking_md/lib/python3.11/site-packages/MDAnalysis/topology/PDBParser.py:295: UserWarning: PDB file is missing resid information.  Defaulted to '1'
  warnings.warn("PDB file is missing resid information.  "
/home/corey/miniconda3/envs/docking_md/lib/python3.11/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"
/tmp/ipykernel_10279/3250331581.py:17: UserWarning: No ligand atoms found in /home/corey/Doc

Loaded 1000 graphs from trajectories.
Loaded 1000 graphs from trajectories


/tmp/ipykernel_10279/3250331581.py:29: UserWarning: Failed to guess bonds for /home/corey/Documents/comp_chem/ml/pose_rank_model/data_trajs/Thrombin/9R8Q/output/topology.pdb: vdw radii for types: Cl. These can be defined manually using the keyword 'vdwradii'
  warnings.warn(f"Failed to guess bonds for {pdb_path}: {e}")
/tmp/ipykernel_10279/3250331581.py:129: UserWarning: Missing topology or trajectory for 6YSJ; skipping.
  warnings.warn(f"Missing topology or trajectory for {pdb_struct}; skipping.")


## Model architecture.

In [25]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class PoseScoringGNN(torch.nn.Module):
    def __init__(self, num_atom_types=100, embed_dim=64, hidden_dim=64, n_outputs=3):
        super().__init__()
        self.embedding = torch.nn.Embedding(num_atom_types, embed_dim)
        self.conv1 = GCNConv(embed_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.lin1 = torch.nn.Linear(hidden_dim, hidden_dim // 2)
        self.lin2 = torch.nn.Linear(hidden_dim // 2, n_outputs)

    def forward(self, data):
        x, edge_index, batch = data.z, data.edge_index, data.batch
        x = self.embedding(x)  # [num_nodes, embed_dim]

        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))

        x = global_mean_pool(x, batch)  # [batch_size, hidden_dim]
        x = F.relu(self.lin1(x))
        out = self.lin2(x)  # [batch_size, n_outputs]
        return out


## Data loading and setup for training.

In [26]:
import numpy as np

# --- Define number of outputs (e.g., RMSD, hydrophobic, polar) ---

# # --- Attach mock labels directly to each graph ---
# for g in graphs:
#     # Each graph gets its own label vector, shape [3]
#     g.y = torch.tensor(np.random.rand(1, n_outputs), dtype=torch.float32)
print(f"Example graph label y: {graphs[0].y}")

# --- Split into train and validation sets ---
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split

# ============================================================
# 1️⃣ Normalize labels before splitting
# ============================================================

# --- Stack all y vectors ---
all_y = np.stack([g.y.squeeze(0).numpy() for g in graphs])

# --- Compute feature-wise mean and std ---
y_mean = all_y.mean(axis=0)
y_std = all_y.std(axis=0) + 1e-8  # prevent divide-by-zero

print("Target means:", y_mean)
print("Target stds:", y_std)

# --- Normalize and assign back ---
for g in graphs:
    g.y = torch.tensor(((g.y.squeeze(0).numpy() - y_mean) / y_std),
                       dtype=torch.float32).unsqueeze(0)

# ============================================================
# 2️⃣ Split into train/val and build loaders
# ============================================================

train_size = int(0.8 * len(graphs))
val_size = len(graphs) - train_size
train_dataset, val_dataset = random_split(graphs, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


Example graph label y: tensor([[ 8.5642e-08, -1.4850e-06,  0.0000e+00]])
Target means: [ 1.5867776  -0.04153871  0.02565144]
Target stds: [0.6087078  0.14070985 0.10919389]


## Quick sanity checks of trajectory graphs.

In [27]:
for g in graphs:
    if g.edge_index.max() >= g.num_nodes or g.edge_index.min() < 0:
        print("⚠️ Invalid edge index found!")
    else:
        assert g.edge_index.shape[0] == 2, "Edge index must be shape [2, num_edges]"
        assert g.z.shape[0] == g.pos.shape[0], "Node feature mismatch"


In [28]:
from torch_geometric.loader import DataLoader

sample = next(iter(train_loader))
print("Sample batch:")
print("pred target shape:", getattr(sample, 'y', None))
print("z shape:", sample.z.shape)
print("pos shape:", sample.pos.shape)
print("edge_index shape:", sample.edge_index.shape)


Sample batch:
pred target shape: tensor([[ 1.8145,  0.2952, -0.2349],
        [-1.7304, -3.0251,  6.4903],
        [-1.0445,  0.2950, -0.2349],
        [ 1.1348,  0.2952, -0.2349],
        [-0.7791,  0.2952, -0.2349],
        [ 0.4349,  0.2952, -0.2349],
        [ 0.3094,  0.2952, -0.2349],
        [ 2.0140,  0.2952, -0.2349],
        [-0.0753,  0.2951, -0.2349],
        [-0.8807,  0.2952, -0.2349],
        [-1.2323,  0.2952, -0.2349],
        [-0.5093,  0.2952, -0.2349],
        [-0.5401,  0.2952, -0.2349],
        [-0.3698,  0.2952, -0.2349],
        [-1.1738,  0.2949, -0.2349],
        [-0.3137,  0.2952, -0.2349],
        [ 0.0566,  0.2952, -0.2349],
        [ 0.7739,  0.2952, -0.2349],
        [-0.7440,  0.2952, -0.2349],
        [-0.7508,  0.2952, -0.2349],
        [-0.0481,  0.2952, -0.2349],
        [ 0.6762,  0.2952, -0.2349],
        [ 1.4934,  0.2952, -0.2349],
        [ 1.9942,  0.2952, -0.2349],
        [ 1.7514,  0.2952, -0.2349],
        [ 1.8705,  0.2952, -0.2349],
     

In [33]:
try:
    model
except NameError:
    model = PoseScoringGNN(n_outputs=1).to(device)

# Quick shape diagnostic
batch = next(iter(train_loader))
batch = batch.to(device)
pred = model(batch)

print(f"pred shape: {pred.shape}")
print(f"batch.y shape: {batch.y.shape}")

pred shape: torch.Size([32, 3])
batch.y shape: torch.Size([32, 3])


## Training loop

In [34]:
# ============================================================
# 3️⃣ Define helper functions for training/evaluation
# ============================================================

def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)
        loss = criterion(pred, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch)
            loss = criterion(pred, batch.y)
            total_loss += loss.item()
            all_preds.append(pred.cpu())
            all_targets.append(batch.y.cpu())
    return total_loss / len(loader), torch.cat(all_preds), torch.cat(all_targets)

In [35]:
def denormalize(y, mean, std):
    """Return y in original physical units"""
    return y * std + mean

In [36]:
from sklearn.metrics import mean_squared_error, r2_score

# ============================================================
# 4️⃣ Training loop
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n_outputs = 3
model = PoseScoringGNN(num_atom_types=100, embed_dim=64, hidden_dim=64, n_outputs=n_outputs).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()

epochs = 20
for epoch in range(epochs):
    train_loss = train(model, train_loader, optimizer, criterion, device)
    val_loss, _, _ = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1:02d} - Train loss: {train_loss:.4f}, Val loss: {val_loss:.4f}")

torch.save(model.state_dict(), "pose_scoring_gnn.pth")

Epoch 01 - Train loss: 0.9830, Val loss: 1.0695
Epoch 02 - Train loss: 0.9759, Val loss: 1.0715
Epoch 03 - Train loss: 0.9666, Val loss: 1.0638
Epoch 04 - Train loss: 0.9421, Val loss: 1.0311
Epoch 05 - Train loss: 0.9176, Val loss: 1.0042
Epoch 06 - Train loss: 0.9052, Val loss: 1.0089
Epoch 07 - Train loss: 0.9020, Val loss: 1.0063
Epoch 08 - Train loss: 0.9044, Val loss: 1.0155
Epoch 09 - Train loss: 0.9049, Val loss: 0.9979
Epoch 10 - Train loss: 0.9046, Val loss: 1.0041
Epoch 11 - Train loss: 0.9026, Val loss: 1.0137
Epoch 12 - Train loss: 0.9027, Val loss: 1.0027
Epoch 13 - Train loss: 0.9047, Val loss: 0.9988
Epoch 14 - Train loss: 0.9040, Val loss: 1.0047
Epoch 15 - Train loss: 0.9020, Val loss: 0.9912
Epoch 16 - Train loss: 0.9057, Val loss: 0.9958
Epoch 17 - Train loss: 0.9037, Val loss: 1.0017
Epoch 18 - Train loss: 0.9042, Val loss: 1.0019
Epoch 19 - Train loss: 0.9012, Val loss: 0.9987
Epoch 20 - Train loss: 0.9008, Val loss: 0.9961


# Eval

In [37]:
# ============================================================
# 5️⃣ Post-training evaluation (denormalized)
# ============================================================

val_loss, all_preds, all_targets = evaluate(model, val_loader, criterion, device)

# --- Move to numpy ---
all_preds = all_preds.numpy()
all_targets = all_targets.numpy()

# --- Denormalize ---
all_preds = denormalize(all_preds, y_mean, y_std)
all_targets = denormalize(all_targets, y_mean, y_std)

# --- Per-output metrics ---
metric_names = ["ΔRMSD (Å)", "ΔE proxy", "f_native"]
for i, name in enumerate(metric_names):
    mse = mean_squared_error(all_targets[:, i], all_preds[:, i])
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(all_preds[:, i] - all_targets[:, i]))
    r2 = r2_score(all_targets[:, i], all_preds[:, i])
    print(f"\n{name}:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  R²:   {r2:.4f}")

# ============================================================
# 6️⃣ Diagnostic parity plots
# ============================================================

# import os
# import matplotlib.pyplot as plt
# import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Directory to save plots
save_dir = "_images"
os.makedirs(save_dir, exist_ok=True)

for i, name in enumerate(metric_names):
    # Ensure NumPy arrays
    y_true = np.array(all_targets)[:, i]
    y_pred = np.array(all_preds)[:, i]

    # --- Fit linear regression line (predicted vs true) ---
    reg = LinearRegression().fit(y_true.reshape(-1, 1), y_pred)
    y_fit = reg.predict(y_true.reshape(-1, 1))
    r2_local = r2_score(y_true, y_pred)
    slope, intercept = reg.coef_[0], reg.intercept_

    # --- Plot parity ---
    plt.figure(figsize=(5, 5))
    plt.scatter(y_true, y_pred, alpha=0.6, edgecolor='k', linewidth=0.3)
    plt.plot(
        [y_true.min(), y_true.max()],
        [y_true.min(), y_true.max()],
        'r--', lw=2, label='Ideal'
    )
    plt.plot(
        y_true, y_fit,
        'b-', lw=1.5, label=f'Fit: y = {slope:.2f}x + {intercept:.2f}'
    )

    # --- Labeling and formatting ---
    plt.xlabel(f"True {name}")
    plt.ylabel(f"Predicted {name}")
    plt.title(f"Parity Plot for {name}")
    plt.legend()
    plt.grid(True)
    plt.text(
        0.05, 0.95,
        f"R² = {r2_local:.3f}",
        transform=plt.gca().transAxes,
        fontsize=10,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
    )
    plt.tight_layout()

    # --- Save to file ---
    fname = os.path.join(save_dir, f"parity_{name.replace(' ', '_')}.png")
    plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.close()

print(f"Saved diagnostic plots (with trendlines and R²) to: {os.path.abspath(save_dir)}")


ΔRMSD (Å):
  RMSE: 0.5555
  MAE:  0.4525
  R²:   0.1566

ΔE proxy:
  RMSE: 0.1453
  MAE:  0.0794
  R²:   0.0131

f_native:
  RMSE: 0.1136
  MAE:  0.0510
  R²:   0.0646
Saved diagnostic plots (with trendlines and R²) to: /home/corey/Documents/comp_chem/ml/pose_rank_model/_images


# Score a pose.

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PoseScoringGNN(
    num_atom_types=100,
    embed_dim=64,
    hidden_dim=64,
    n_outputs=3  # same as used during training
).to(device)

model.load_state_dict(torch.load("pose_scoring_gnn.pth", map_location=device))
model.eval()


PoseScoringGNN(
  (embedding): Embedding(100, 64)
  (conv1): GCNConv(64, 64)
  (conv2): GCNConv(64, 64)
  (conv3): GCNConv(64, 64)
  (lin1): Linear(in_features=64, out_features=32, bias=True)
  (lin2): Linear(in_features=32, out_features=3, bias=True)
)

In [39]:
def load_single_pose_as_graph(pdb_path, ligand_resname="UNK"):
    import MDAnalysis as mda
    from torch_geometric.data import Data
    import torch

    u = mda.Universe(pdb_path)
    ligand = u.select_atoms(f"resname {ligand_resname}")
    if len(ligand) == 0:
        raise ValueError(f"No ligand atoms found in {pdb_path}")

    ligand.guess_bonds()
    atom_index_map = {atom.index: i for i, atom in enumerate(ligand.atoms)}
    edge_index = []

    for bond in ligand.bonds:
        i, j = [atom_index_map[a.index] for a in bond.atoms]
        edge_index.append([i, j])
        edge_index.append([j, i])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    z = torch.tensor([Chem.GetPeriodicTable().GetAtomicNumber(atom.element) for atom in ligand], dtype=torch.long)
    pos = torch.tensor(ligand.positions, dtype=torch.float)

    return Data(z=z, edge_index=edge_index, pos=pos)


In [40]:
graph = load_single_pose_as_graph("example_docked.pdb", ligand_resname="UNK")

In [41]:
from torch_geometric.loader import DataLoader

graph = graph.to(device)
model.eval()

with torch.no_grad():
    pred = model(graph)

# ΔRMSD, ΔE_proxy, f_native
print("Predicted scores:", pred.cpu().numpy())

Predicted scores: [[-0.17350504 -0.02209549  0.08248367]]


Output index	Metric	Interpretation	Desirable direction
* 0	ΔRMSD	RMSD between predicted and native ligand pose (Å)	lower = better
* 1	ΔE_proxy	Approximate interaction energy proxy	lower = stronger binding
* 2	f_native	Fraction of native contacts	higher = better

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "input_dim": input_dim,
    "hidden_dim": hidden_dim,
    "global_feat_dim": global_feat_dim
    }, "class_a_gpcr_pic50_gin.pt")